# 04) Train with W&B

## Setup

Loads `data/train.csv`/`data/columns.json` fresh (same convention as every other notebook here),
logs in to W&B (credentials already cached in `~/.netrc` -- no key ever passed here), and rebuilds
the exact 10-fold expanding-window CV split from `03)`: walk-forward, `MIN_TRAIN_DAYS=7` warm-up,
`DELTA_DAYS=1` step, no lookahead.

The loaded DataFrame is `train_df`, not `train` -- `train` is reserved for the per-model function
each W&B sweep calls.

In [1]:
import json

import numpy as np
import pandas as pd
import wandb
from scipy.stats import spearmanr
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

train_df = pd.read_csv("data/train.csv", parse_dates=["date"])
with open("data/columns.json") as f:
    columns = json.load(f)

TARGET_COL = columns["target_cols"][0]
FINAL_FEATURE_COLS = columns["final_feature_cols"]

wandb.login()

print(f"train_df: {train_df.shape[0]} rows, target = {TARGET_COL!r}")
print(f"{len(FINAL_FEATURE_COLS)} final features:", FINAL_FEATURE_COLS)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/abprada/.netrc.


wandb: Currently logged in as: ablancoprada (ablancoprada-massachusetts-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


train_df: 8443 rows, target = 'target_bps_vs_mid'
4 final features: ['near_price_vs_mid_bps', 'urgency', 'ref_price_vs_mid_bps', 'far_price_vs_mid_bps']


In [2]:
unique_dates = np.sort(train_df["date"].unique())
MIN_TRAIN_DAYS = 7
DELTA_DAYS = 1

folds = []
for fold_idx, eval_start in enumerate(range(MIN_TRAIN_DAYS, len(unique_dates), DELTA_DAYS)):
    fit_dates = unique_dates[:eval_start]
    eval_dates = unique_dates[eval_start:eval_start + DELTA_DAYS]
    assert fit_dates.max() < eval_dates.min(), "eval dates must all fall after fit dates"

    fit_mask = train_df["date"].isin(fit_dates)
    eval_mask = train_df["date"].isin(eval_dates)
    folds.append((fit_mask, eval_mask))

    print(f"fold {fold_idx}: fit {fit_mask.sum():>4} rows ({len(fit_dates)} days, "
          f"{pd.Timestamp(fit_dates.min()).date()} to {pd.Timestamp(fit_dates.max()).date()}) | "
          f"eval {eval_mask.sum():>4} rows ({len(eval_dates)} days, "
          f"{pd.Timestamp(eval_dates.min()).date()} to {pd.Timestamp(eval_dates.max()).date()})")

fold 0: fit 3476 rows (7 days, 2025-03-03 to 2025-03-11) | eval  496 rows (1 days, 2025-03-12 to 2025-03-12)
fold 1: fit 3972 rows (8 days, 2025-03-03 to 2025-03-12) | eval  497 rows (1 days, 2025-03-13 to 2025-03-13)
fold 2: fit 4469 rows (9 days, 2025-03-03 to 2025-03-13) | eval  497 rows (1 days, 2025-03-14 to 2025-03-14)
fold 3: fit 4966 rows (10 days, 2025-03-03 to 2025-03-14) | eval  497 rows (1 days, 2025-03-17 to 2025-03-17)
fold 4: fit 5463 rows (11 days, 2025-03-03 to 2025-03-17) | eval  496 rows (1 days, 2025-03-18 to 2025-03-18)
fold 5: fit 5959 rows (12 days, 2025-03-03 to 2025-03-18) | eval  497 rows (1 days, 2025-03-19 to 2025-03-19)
fold 6: fit 6456 rows (13 days, 2025-03-03 to 2025-03-19) | eval  496 rows (1 days, 2025-03-20 to 2025-03-20)
fold 7: fit 6952 rows (14 days, 2025-03-03 to 2025-03-20) | eval  497 rows (1 days, 2025-03-21 to 2025-03-21)
fold 8: fit 7449 rows (15 days, 2025-03-03 to 2025-03-21) | eval  497 rows (1 days, 2025-03-24 to 2025-03-24)
fold 9: fit 7

## Shared sklearn sweep runner

One `train()`-function factory reused by every sklearn model's sweep (`HuberRegressor`, `Lasso`,
`GradientBoostingRegressor`, `RandomForestRegressor`): builds `impute -> scale -> model`, fits/evals
across all 10 folds, logs the full metric set. `random_state` is fixed to `0` automatically for
models that accept it (not swept -- a determinism knob, not a tunable choice).

In [3]:
def make_sklearn_train_fn(estimator_class):
    def train():
        wandb.init()
        config = dict(wandb.config)
        if "random_state" in estimator_class().get_params():
            config["random_state"] = 0
        estimator = estimator_class(**config)

        model = Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("model", estimator),
        ])

        fit_mses, eval_mses = [], []
        fit_r2s, eval_r2s = [], []
        fit_ics, eval_ics = [], []
        for fit_mask, eval_mask in folds:
            X_fit = train_df.loc[fit_mask, FINAL_FEATURE_COLS]
            y_fit = train_df.loc[fit_mask, TARGET_COL]
            X_eval = train_df.loc[eval_mask, FINAL_FEATURE_COLS]
            y_eval = train_df.loc[eval_mask, TARGET_COL]

            model.fit(X_fit, y_fit)
            pred_fit = model.predict(X_fit)
            pred_eval = model.predict(X_eval)

            fit_mses.append(mean_squared_error(y_fit, pred_fit))
            eval_mses.append(mean_squared_error(y_eval, pred_eval))
            fit_r2s.append(r2_score(y_fit, pred_fit))
            eval_r2s.append(r2_score(y_eval, pred_eval))
            fit_ics.append(spearmanr(y_fit, pred_fit).statistic)
            eval_ics.append(spearmanr(y_eval, pred_eval).statistic)

        wandb.log({
            "fit_mse": np.mean(fit_mses),
            "eval_mse_mean": np.mean(eval_mses),
            "eval_mse_std": np.std(eval_mses),
            "fit_r2": np.mean(fit_r2s),
            "eval_r2": np.mean(eval_r2s),
            "fit_ic": np.mean(fit_ics),
            "eval_ic": np.mean(eval_ics),
        })
        wandb.finish()
    return train

print("make_sklearn_train_fn ready")

make_sklearn_train_fn ready


## HuberRegressor sweep (best linear)

`epsilon` controls where Huber's loss switches from quadratic to linear (robustness to outliers);
`alpha` is the L2 penalty strength. 30-trial Bayesian search, optimizing held-out IC.

In [4]:
from sklearn.linear_model import HuberRegressor

huber_sweep_config = {
    "method": "bayes",
    "metric": {"name": "eval_ic", "goal": "maximize"},
    "parameters": {
        "epsilon": {"distribution": "log_uniform_values", "min": 1.1, "max": 3.0},
        "alpha": {"distribution": "log_uniform_values", "min": 1e-5, "max": 1e1},
    },
}

huber_sweep_id = wandb.sweep(huber_sweep_config, project="close-auction-model-tuning")
wandb.agent(huber_sweep_id, function=make_sklearn_train_fn(HuberRegressor), count=30)
print("huber_sweep_id:", huber_sweep_id)

Create sweep with ID: git37eu2
Sweep URL: https://wandb.ai/ablancoprada-massachusetts-institute-of-technology/close-auction-model-tuning/sweeps/git37eu2


wandb: Agent Starting Run: iyqz2isj with config:


wandb: 	alpha: 0.010484130752918316


wandb: 	epsilon: 2.309743432968326


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/abprada/.netrc.


...

wandb: uploading history steps 0-0, summary


wandb: 
wandb: Run history:
wandb:       eval_ic ▁
wandb: eval_mse_mean ▁
wandb:  eval_mse_std ▁
wandb:       eval_r2 ▁
wandb:        fit_ic ▁
wandb:       fit_mse ▁
wandb:        fit_r2 ▁
wandb: 
wandb: Run summary:
wandb:       eval_ic 0.82061
wandb: eval_mse_mean 9.87588
wandb:  eval_mse_std 2.60005
wandb:       eval_r2 0.81669
wandb:        fit_ic 0.82185
wandb:       fit_mse 13.69423
wandb:        fit_r2 0.81343
wandb: 


wandb: 🚀 View run curious-sweep-30 at: https://wandb.ai/ablancoprada-massachusetts-institute-of-technology/close-auction-model-tuning/runs/fydekz2t
wandb: ⭐️ View project at: https://wandb.ai/ablancoprada-massachusetts-institute-of-technology/close-auction-model-tuning
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260819_024857-fydekz2t/logs


huber_sweep_id: git37eu2
